# Alloy — Single Experiment

> **This notebook is not meant to be run directly.**
> It is executed by `multiple_experiments.ipynb` via papermill, which injects
> the parameters in the **Parameters** cell below.

For a given (query, throughput, parallelism) configuration, this notebook:
1. Connects to an already-provisioned Grid5000 cluster
2. Runs the **vanilla** baseline (plain Flink + Kafka, no proxy)
3. Runs the **Alloy** variant (Flink + Kafka + Envoy proxy with Wasm filter)
4. Collects CPU, throughput, and network metrics from Prometheus
5. Saves results to `results_vanilla.csv` and `results_alloy.csv`

## Imports

In [ ]:
import enoslib as en
import pandas as pd

## Parameters

These variables are **injected by papermill** when this notebook is executed by
`multiple_experiments.ipynb`. Default values are used when running manually.

| Parameter | Description |
|---|---|
| `query` | SQL filename from `config/` (e.g. `q1.sql`) |
| `sources` | Nexmark event types consumed by the query (e.g. `["bid"]`) |
| `throughput` | Target event injection rate (events/s) |
| `parallelism` | Number of Flink task managers and Kafka partitions |
| `duration` | Measurement window in seconds |
| `output_dir` | Directory where result CSVs and the executed notebook are saved |
| `init_monitoring` | Set to `False` when called from `multiple_experiments.ipynb` (cluster already provisioned) |
| `cluster` | Grid5000 cluster name |
| `max_parallelism` | Total parallelism reserved on the cluster |

In [ ]:
compose_file = "./docker-compose.yml" 
envoy_config = "envoy-alloy-test1.yml"
envoy_config_file = "./" + envoy_config  
envoy_version = "latest"
df_metric = None
parallelism = 1
max_parallelism = 1
nb_partitions = parallelism
duration=300
max_quantity=0
cluster = "ecotype"

throughput=30000

nb_tm=2
nb_ts=1
nb_cpu = nb_ts 
fetchMinBytes = 100000

query="q0.sql"
sources = ["bid"]

debug=True

output_dir = "/home/dschmitz/alloy/xp/macro/full_stack"
cur_dir = output_dir
init_monitoring = True

## Environment

Docker Swarm environment variables passed to the stack deployment.

In [ ]:
env = {
    "NUM_SOURCES": parallelism,
    "NUM_PARTITIONS": nb_partitions,
    "NUM_TM": parallelism,
    "NUM_TS": 4,
    "NUM_CPU": nb_ts + 1,
    "ENVOY_VERSION": envoy_version,
    "ENVOY_CONFIG": envoy_config_file,
    #"DEBUG" =  "rest.flamegraph.enabled: true",
}

## Cluster Connection

Connects to the Grid5000 reservation and initialises the cluster.
When `init_monitoring=False` (set by `multiple_experiments.ipynb`), the Docker and
monitoring setup is skipped — the cluster was already initialised by the campaign notebook.

In [ ]:
import sys
sys.path.insert(0, f"{cur_dir}/../infra/grid5000")
job_name = "alloy-tests"

import init
role_name="control"
pattern_manager="control[0]"

conf = (en.G5kConf.from_settings(job_name=job_name, walltime="00:30:00", job_type=[])
        .add_machine(roles=["manager", "control", "injector"], cluster=cluster, nodes=1)
        .add_machine(roles=["control","jobmanager"], cluster=cluster, nodes=1)
        .add_machine(roles=["control","taskmanager"], cluster=cluster, nodes=max_parallelism))

for i in range(max_parallelism):
    conf.add_machine(roles=["control",f"kafka{i+1}"], cluster=cluster, nodes=1)
    
local_conf = (
    conf.finalize()
)

provider = en.G5k(local_conf)
roles, networks = provider.init()

display(roles)
xp = init.AlloyG5KExperimentation(roles, pattern_manager)

if init_monitoring:
    xp.init_docker_swarm()
    xp.init_docker_nodes()
    with en.actions(roles=roles) as a:
        a.apt(name="python3-pip", state="present", update_cache="true")
        a.pip(name="docker<7.1.0", state="present")
        a.pip(name="requests<2.32", state="present")
    xp.init_docker_network()
    xp.init_monitoring()
    xp.load_local_images(f"{cur_dir}/../images")

## Setup

Uploads the `config/` directory (SQL queries, Docker Compose templates, Envoy config)
to the manager node.

In [ ]:
with en.play_on(roles=xp.roles, pattern_hosts=xp.pattern_manager) as t:
    t.copy(src=f"{cur_dir}/config/", dest="/root/config/", mode="0755")

## Stack Helpers

`deploy_stack` deploys the Flink + Kafka (+Envoy) Docker Swarm stack with the
environment variables for the current parallelism level.

`deploy_generator` starts the Nexmark event generator (Berserker) as a Flink job,
injecting bid/auction/person events at the configured `throughput` rate into Kafka.

In [ ]:
def deploy_stack(compose_file="./config/docker-composer.yml"):
    ###
    env = {
        "NUM_SOURCES": parallelism,
        "NUM_PARTITIONS": nb_partitions,
        "NUM_TM": parallelism,
        "NUM_TS": nb_ts,
        "NUM_CPU": nb_ts + 1,
        #"DEBUG" =  "rest.flamegraph.enabled: true",
    }
    
    xp.deploy_swarm(compose_file, env=env, name="test", wait=0)

In [ ]:
def deploy_generator():
    generator_compose = f"{cur_dir}/config/docker-compose-generator.yml"
    xp.deploy_swarm(generator_compose, name="generator")
    
    job_name = "alloy-tests"
    
    params = {
        "--jobName" : f"generator_{parallelism}_raw",
        "--parallelism": f"{parallelism}",
        "--p-bid-source": f"{parallelism}",
        "--p-auction-source": f"{parallelism}",
        "--p-person-source": f"{parallelism}",
        "--auctionRate": f"{int(throughput*0.06) if 'auction' in sources else 0}",
        "--bidRate": f"{int(throughput*0.92) if 'bid' in sources else 0}",
        "--personRate": f"{int(throughput*0.02) if 'person' in sources else 0}",
        "--kafkaProducerAddress": "kafka-edge1:9092",
    }
    
    generator_job_id = xp.deploy_job(f"{cur_dir}/benchmarks/flink-examples/target/flink-examples.jar", "ch.ethz.systems.strymon.ds2.flink.nexmark.generator.Generator", args=params, port=18081)

## Vanilla Baseline

Deploys the plain Flink + Kafka stack (no Alloy proxy), starts the generator,
and runs the query via the Flink Table API.
Metrics are collected for `duration` seconds then saved to `results_vanilla.csv`.
The first 30 seconds are discarded to exclude the warm-up phase.

In [ ]:

vanilla_compose_template = f"{cur_dir}/config/docker-compose-vanilla-template.yml"
vanilla_compose = f"{output_dir}/docker-compose-vanilla.yml"
xp.generate_config_files(vanilla_compose_template, None, parallelism, output_file_name=vanilla_compose)
deploy_stack(vanilla_compose)
deploy_generator()
df = xp.launch_table_api_run(job_name="vanilla", duration=duration, alloy=False, query=query, parallelism=parallelism, disable_operator_chaining=True)
df = df[df.index > pd.Timedelta(seconds=30)]
df.to_csv(f"{output_dir}/results_vanilla.csv")

## Alloy Run

1. **Compile**: runs the control plane in `COMPILE` mode to extract the Alloy plan
   (`alloy_plan.yaml`) from the SQL query using Flink's Table API and Apache Calcite.
2. **Configure proxies**: renders the Envoy config template (`envoy-alloy-test.yml.j2`)
   for each Kafka broker, injecting the selections and projections from the Alloy plan.
3. **Deploy**: starts the Alloy stack (Flink + Kafka + Envoy with Wasm filter) and the generator.
4. **Measure**: runs the query via the control plane in `RUN_ALLOY` mode, collects metrics
   for `duration` seconds, and saves to `results_alloy.csv`.

In [ ]:
import jinja2
import yaml

###### Prepare ######
xp.compile_alloy(query=query, output_dir=output_dir, parallelism=parallelism, disable_operator_chaining=True)

# Open the file in read mode
file = open(f"{cur_dir}/config/envoy-alloy-test.yml.j2", "r")
# Read the entire content of the file
template_str = file.read()

with open(f"{output_dir}/config/alloy_plan.yaml", "r") as f:
    envoy_yaml = yaml.safe_load(f)

# Create the Jinja2 template
for i in range(parallelism):
    template = jinja2.Template(template_str)
    data = { 
        "alloy_filters": yaml.dump(envoy_yaml),
        "envoy_id": f"envoy-mqtt-edge{i + 1}",
        "kafka_address": f"kafka-edge{i + 1}"
    }
    # Render the template with the provided variables
    result = template.render(**data)
    with open(f"{output_dir}/config/envoy-alloy-test{i + 1}.yml", "w+") as f:
        f.write(result)
    
    with en.play_on(roles=xp.roles, pattern_hosts=xp.pattern_manager) as t:
        t.copy(src=f"{output_dir}/config/envoy-alloy-test{i + 1}.yml", dest="./", mode="0755")
    
alloy_compose_template = f"{cur_dir}/config/docker-compose-alloy-template.yml"
alloy_compose = f"{output_dir}/docker-compose-alloy.yml"

xp.generate_config_files(alloy_compose_template, f"{output_dir}/config/envoy-alloy-test1.yml", parallelism, output_file_name=alloy_compose)
env = {
        "NUM_SOURCES": parallelism,
        "NUM_PARTITIONS": parallelism,
        "NUM_TM": parallelism,
        "NUM_TS": nb_ts,
        "NUM_CPU": nb_ts + 1,
    }
xp.deploy_swarm(alloy_compose, name="test", env=env, wait=0)
deploy_generator()

###### Run ######
df = xp.launch_table_api_run(job_name="alloy", duration=duration, alloy=True, query=query, parallelism=parallelism, disable_operator_chaining=True)
df = df[df.index > pd.Timedelta(seconds=30)]
df.to_csv(f"{output_dir}/results_alloy.csv")

## Teardown

Removes all Docker Swarm stacks (`test`, `generator`, `alloy`) from the cluster.

In [ ]:
xp.remove_stack(["test", "generator", "alloy"])